# TwoTank - Plot Evaluation Results

Load nominal trajectories and select one to plot.


In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')
def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / "src" / "sdpc").is_dir() and (candidate / "case_studies").is_dir():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the SD-DPC repository")

REPO_ROOT = find_repo_root()
CASE_DIR = REPO_ROOT / "case_studies" / "TwoTank"

sys.path.insert(0, str(REPO_ROOT / "src"))

import matplotlib.pyplot as plt
import pandas as pd
import torch

from sdpc.config import load_config, seed_everything
from sdpc.plotting import plot_states_and_controls
from sdpc.registry import make_system

device = torch.device('cpu')
system = make_system('twotank', device=device)
CONFIGS = CASE_DIR / 'configs'
RESULTS = CASE_DIR / 'results'


## Select result and trajectory

Set both values and rerun this cell.


In [ ]:
RESULT_KIND = 'nominal'
TRAJECTORY_INDEX = 3
FONT_SIZE = 12.0
LEGEND_FONT_SIZE = 13.0
TRAJECTORY_LINEWIDTHS = {
    'sd_dpc': 2.0, 'nn_dpc': 1.4, 'mpc': 1.4,
}
TRAJECTORY_ALPHAS = {
    'sd_dpc': 1.0,
    'nn_dpc': 0.7,
    'mpc': 0.85,
}
SHOW_CONTROL_RATE = True
LEGEND_LOC = 'lower center'
LEGEND_BBOX = (0.5, 1.0)  # axes coordinates; adjust to move the top legend
LEGEND_NCOL = 4

config_name = 'eval_nominal.yaml'
cfg = load_config(CONFIGS / config_name)
seed_everything(int(cfg.get('seed', 0)))
result_dir = (
    RESULTS / 'eval'
    / f"{RESULT_KIND}_{cfg['n_trajectories']}x{cfg['nsteps']}"
)
payload = torch.load(result_dir / 'all_trajectories.pt', map_location=device)
trajectories = payload['trajectories']
scenarios = payload['scenarios']
n_trajectories = scenarios['xn'].shape[0]
if not 0 <= TRAJECTORY_INDEX < n_trajectories:
    raise IndexError(f'TRAJECTORY_INDEX must be in [0, {n_trajectories - 1}]')

styles = {
    'sd_dpc': ('SD-DPC', '#1F497D', '-'),
    'nn_dpc': ('NN-DPC', '#2D6A4F', '--'),
    'mpc': ('MPC', '#E69F00', '-.'),
}
legend_order = ['Reference', 'SD-DPC', 'NN-DPC', 'MPC']
xmin, xmax = system.xmin, system.xmax
state_margin = None

fig, axes = plot_states_and_controls(
    [
        {
            'x': trajectories[name]['x_traj'][TRAJECTORY_INDEX:TRAJECTORY_INDEX + 1],
            'u': trajectories[name]['u_traj'][TRAJECTORY_INDEX:TRAJECTORY_INDEX + 1],
            'label': label,
            'color': color,
            'linestyle': linestyle,
            'alpha': TRAJECTORY_ALPHAS.get(name, 1.0),
            'linewidth': TRAJECTORY_LINEWIDTHS.get(name, 2.0),
        }
        for name, (label, color, linestyle) in styles.items()
    ],
    r_traj=scenarios['r'][TRAJECTORY_INDEX:TRAJECTORY_INDEX + 1],
    xmin=xmin,
    xmax=xmax,
    umin=system.umin,
    umax=system.umax,
    state_margin=state_margin,
    title='',
    publication=True,
    figsize=(6, 6 if SHOW_CONTROL_RATE else 4.2),
    show_control_rate=SHOW_CONTROL_RATE,
    show_legend=True,
    show_state_bounds=True,
    show_control_bounds=True,
    font_size=FONT_SIZE,
    legend_loc=LEGEND_LOC,
    legend_bbox=LEGEND_BBOX,
    legend_ncol=LEGEND_NCOL,
    legend_fontsize=LEGEND_FONT_SIZE,
    legend_order=legend_order,
    use_tex=True,
)

figure_dir = result_dir / 'figures'
figure_dir.mkdir(parents=True, exist_ok=True)
figure_path = figure_dir / f'trajectory_{TRAJECTORY_INDEX}.pdf'
fig.savefig(figure_path, bbox_inches='tight')
plt.show()

rows = pd.read_csv(result_dir / 'per_trajectory.csv')
display(rows[rows['trajectory'] == TRAJECTORY_INDEX].set_index('method'))
print(f'Plotted {RESULT_KIND} trajectory {TRAJECTORY_INDEX} of {n_trajectories - 1}')
print('Saved:', figure_path)
